# DMRG Ground-State Energy — Quickstart

This notebook shows the fastest path to computing molecular ground-state energies
with the Qumulator **DMRG engine** via the SDK.

The `client.dmrg.energy()` method:
- Accepts one-body (`h1e`) and two-body (`h2e`) integrals in the active-space basis
- Runs two-site DMRG sweeps with controllable bond dimension (`d_max`)
- Returns energy, convergence status, and timing

**Documentation**: https://sdk.qumulator.com/docs/dmrg

---
Install:
```bash
pip install qumulator-sdk pyscf
```

In [ ]:
from qumulator import QumulatorClient
from pyscf import gto, scf, mcscf, ao2mo
import numpy as np

client = QumulatorClient()  # reads QUMULATOR_API_URL + QUMULATOR_API_KEY from env

## Step 1: Prepare integrals with PySCF

In [ ]:
mol = gto.M(atom="H 0 0 0; H 0 0 0.74", basis="sto-3g", spin=0, verbose=0)
mf  = scf.RHF(mol); mf.verbose = 0; mf.run()

mc  = mcscf.CASSCF(mf, ncas=2, nelecas=2); mc.verbose = 0; mc.run()

h1e, e_core = mc.get_h1eff()
h2e  = ao2mo.restore(1, mc.get_h2eff(), mc.ncas)
e_nuc = mc.energy_nuc() + e_core

print(f"h1e shape: {h1e.shape}   h2e shape: {h2e.shape}")

## Step 2: Call the DMRG engine

In [ ]:
result = client.dmrg.energy(
    h1e=h1e.tolist(),
    h2e=h2e.tolist(),
    n_elec=[1, 1],          # (n_alpha, n_beta)
    e_nuc=float(e_nuc),
    d_max=64,               # bond dimension — higher = more accurate
    n_sweeps=8,             # number of DMRG sweeps
)

print(f"E(DMRG)       = {result.energy:.10f} Ha")
print(f"converged     = {result.converged}")
print(f"sweeps run    = {result.n_sweeps_run}")
print(f"d_max used    = {result.d_max_used}")
print(f"wall time     = {result.wall_time_s:.3f} s")

## Step 3: Compare with PySCF FCI

In [ ]:
from pyscf import fci
fcisolver = fci.FCI(mol, mf.mo_coeff); fcisolver.verbose = 0
E_fci, _ = fcisolver.kernel()

print(f"E(FCI)        = {E_fci:.10f} Ha")
print(f"|DMRG − FCI|  = {abs(result.energy - E_fci):.2e} Ha")
print(f"Chemical acc  = {abs(result.energy - E_fci) < 1.6e-3}  (< 1.6 mHa threshold)")

## Step 4: Choosing d_max and n_sweeps

| Active space | Recommended d_max | Sweeps | Typical time |
|---|---|---|---|
| ≤ 6 orbitals | 16–32 | 8 | < 1 s |
| 6–12 orbitals | 64 | 8–12 | 1–30 s |
| 12–20 orbitals | 128–256 | 10–20 | 30 s – 5 min |
| 20–30 orbitals | 256–512 | 15–50 | 5–30 min |

Convergence is reached when |ΔE| < `tol` (default 10⁻¹⁰ Ha) between sweeps.

For molecules where `n_sweeps` is insufficient, DMRG returns `converged=True` conservatively
(the algorithm breaks early when ΔE < tol). Increase `n_sweeps` if you need to verify
convergence rigorously.

In [ ]:
# Larger active space example — H2O CAS(4,6)
from pyscf import gto, scf, mcscf, ao2mo

mol2 = gto.M(
    atom="O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469",
    basis="sto-3g", spin=0, verbose=0
)
mf2 = scf.RHF(mol2); mf2.verbose = 0; mf2.run()
mc2 = mcscf.CASSCF(mf2, ncas=4, nelecas=(3,3)); mc2.verbose = 0; mc2.run()

h1e2, ec2 = mc2.get_h1eff()
h2e2 = ao2mo.restore(1, mc2.get_h2eff(), mc2.ncas)
en2 = mc2.energy_nuc() + ec2

r2 = client.dmrg.energy(
    h1e=h1e2.tolist(), h2e=h2e2.tolist(),
    n_elec=[3, 3], e_nuc=float(en2),
    d_max=64, n_sweeps=10,
)
print(f"H₂O CAS(4,6) DMRG: E = {r2.energy:.8f} Ha  converged={r2.converged}  t={r2.wall_time_s:.1f}s")

## Result fields

| Field | Type | Description |
|---|---|---|
| `energy` | float | Ground-state energy in Hartrees (active-space + nuclear) |
| `converged` | bool | True if DMRG converged within tolerance |
| `n_sweeps_run` | int | Number of sweeps actually performed |
| `d_max_used` | int | Bond dimension used |
| `n_orb` | int | Number of active orbitals |
| `n_so` | int | Number of spin-orbitals = 2 × n_orb |
| `wall_time_s` | float | Server-side wall time in seconds |

See full docs at [sdk.qumulator.com/docs/dmrg](https://sdk.qumulator.com/docs/dmrg).